# Choosing between asyncio, threading, and multiprocessing — benchmarks and decision guide

last_verified: 2026-10-03 · tool_version: 3.13.5

## Purpose

Compare the three standard-library concurrency models on representative workloads, measure wall-clock cost directly in the cells below, and leave a repeatable rule for picking one.

## When to use

- Reach for `asyncio` when the workload waits on many concurrent IO operations with an async API available.
- Reach for `threading` (via `ThreadPoolExecutor`) when the workload blocks on calls that expose no async API, or when threads must share in-memory state.
- Reach for `multiprocessing` (via `ProcessPoolExecutor`) when the workload is CPU-bound and must scale past a single interpreter.

## Prerequisites

- The local Python interpreter (timings below were measured against it; rerun the cells on any machine to get local numbers).
- Standard library only: `asyncio`, `concurrent.futures`, `time`. No third-party packages.

In [ ]:
import asyncio
import math
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

IO_TASKS = 8
IO_DELAY = 0.05


def timed(label, fn):
    """Run fn, print wall-clock seconds, return (result, elapsed)."""
    start = time.perf_counter()
    result = fn()
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.3f}s")
    return result, elapsed


## Steps

1. Run the IO-bound cell: identical sleep workloads under sequential, threads, and asyncio.
2. Run the CPU-bound cell: identical arithmetic chunks under sequential and processes, with a result-equality assert.
3. Read the decision table, then run the `choose_model` checks.

| Workload shape | Pick | Reason |
|---|---|---|
| Many concurrent IO waits, async API exists | `asyncio` | One thread, one event loop; cheapest per-task overhead |
| Blocking calls with no async API | `threading` | Threads block independently; pool bounds concurrency |
| CPU-bound number crunching | `multiprocessing` | Separate processes scale past one interpreter |
| Shared in-memory state, light contention | `threading` | Threads share memory; processes would need explicit IPC |

In [ ]:
def io_sequential():
    for _ in range(IO_TASKS):
        time.sleep(IO_DELAY)


def io_threads():
    with ThreadPoolExecutor(max_workers=IO_TASKS) as pool:
        list(pool.map(time.sleep, [IO_DELAY] * IO_TASKS))


async def _one_async():
    await asyncio.sleep(IO_DELAY)


async def _io_async():
    await asyncio.gather(*(_one_async() for _ in range(IO_TASKS)))


def io_asyncio():
    asyncio.run(_io_async())


for _label, _fn in [("sequential", io_sequential), ("threads", io_threads), ("asyncio", io_asyncio)]:
    timed(_label, _fn)


In [ ]:
# math.factorial is used as the CPU workload because it is importable
# by worker processes; interactively defined functions cannot be pickled.
WORK = [100000] * 8


def cpu_sequential():
    return [math.factorial(n) for n in WORK]


def cpu_processes():
    with ProcessPoolExecutor() as pool:
        return list(pool.map(math.factorial, WORK))


seq, _t_seq = timed("sequential", cpu_sequential)
par, _t_par = timed("processes", cpu_processes)
assert seq == par, "parallel result must match sequential result"
print(f"results identical: {seq == par}")


In [ ]:
def choose_model(*, io_bound, needs_shared_memory=False, task_count=1):
    """Return the recommended model for a workload description."""
    if io_bound and task_count >= 2:
        return "asyncio"
    if io_bound:
        return "threading"
    if needs_shared_memory:
        return "threading"
    return "multiprocessing"


checks = [
    (dict(io_bound=True, task_count=50), "asyncio"),
    (dict(io_bound=True, task_count=1), "threading"),
    (dict(io_bound=False), "multiprocessing"),
    (dict(io_bound=False, needs_shared_memory=True), "threading"),
]
for kwargs, expected in checks:
    assert choose_model(**kwargs) == expected, kwargs
print(f"decision checks passed: {len(checks)}")


## Verify

- IO cell: threaded and asyncio runs complete in roughly one sleep interval while sequential takes the sum; exact ratios vary by machine, the ordering is what matters.
- CPU cell: the `results identical: True` line confirms the process pool computes the same values as the loop.
- Decision cell: `decision checks passed: 4` confirms the table rule is encoded correctly.

## Common errors

- Calling `asyncio.sleep` without `await` inside a coroutine: the sleep never happens and the gather finishes instantly with a warning.
- Passing a lambda, closure, or notebook-defined helper to `ProcessPoolExecutor.map`: pool workers must pickle the function by qualified name, so only importable callables (like `math.factorial` above) work.
- Using threads to speed up pure arithmetic: threads share one interpreter, so CPU-bound loops gain nothing; move that work to processes.
- Spawning more processes than cores for small chunks: pickling and startup dominate, and the pool run is slower than the loop.

## References

- Standard-library modules exercised here: `asyncio`, `threading`, `multiprocessing`, `concurrent.futures`, `time`.
- Sibling comparison notebook in this directory covers async-runtime choice (`async-patterns-comparison.ipynb`); this guide covers the concurrency-model choice one level up.